# Day 067 — Exercise 3: Classify Image

**What you'll build:** `classify_image(img_b64, labels, describe_fn=None)` — zero-shot image classification by prompting a vision LLM.

**Why it matters:** Zero-shot classification needs no labelled training data. You describe the categories in the prompt and the model applies its visual knowledge. This pattern is used for product categorisation, content moderation, and medical image triage.

In [ ]:
import io
import base64
from PIL import Image

def image_to_base64(img: Image.Image, format: str = 'PNG') -> str:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

import ollama  # imported so the stub compiles; not called in checks

def describe_image(img_b64: str, prompt: str = 'Describe this image.',
                   describe_fn=None) -> str:
    if describe_fn is not None:
        return describe_fn(img_b64, prompt)
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
    )
    return resp['message']['content']

_test_img = Image.new('RGB', (100, 100), color=(50, 200, 50))
_test_b64 = image_to_base64(_test_img)


## Task

Implement `classify_image(img_b64, labels, describe_fn=None) -> str`:

1. Build a prompt listing all labels and asking for exactly one
2. Call `describe_image(img_b64, prompt, describe_fn=describe_fn)`
3. Search `response.lower()` for each `label.lower()` — return first match
4. Fall back to `labels[0]` if no label found in the response

## Your Implementation

In [ ]:
def classify_image(img_b64: str, labels: list,
                   describe_fn=None) -> str:
    """Classify an image into one of the given labels using a vision LLM.

    Builds a classification prompt that asks the model to pick one label.
    Parses the response to find which label appears (case-insensitive).
    Falls back to labels[0] if no label found in the response.

    Args:
        img_b64:     base64-encoded image string
        labels:      list of category strings, e.g. ['cat', 'dog', 'other']
        describe_fn: callable(img_b64, prompt) -> str for testing
    Returns:
        One of the strings from labels
    """
    raise NotImplementedError


In [ ]:
def classify_image(img_b64: str, labels: list,
                   describe_fn=None) -> str:
    label_list = ', '.join(f'"{l}"' for l in labels)
    prompt = (
        f'Classify this image into exactly one of these categories: '
        f'{label_list}. Reply with only the category name, nothing else.'
    )
    response = describe_image(img_b64, prompt, describe_fn=describe_fn)
    resp_lower = response.lower()
    for label in labels:
        if label.lower() in resp_lower:
            return label
    return labels[0]


## Automated checks

In [ ]:
score, total = 0, 5
try:
    labels = ['red', 'green', 'blue']

    # Mock returns 'green' — should match the second label
    result = classify_image(_test_b64, labels,
                            describe_fn=lambda b, p: 'green')
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    score += 1; print("\u2705 returns a string")

    assert result == 'green', f"Expected 'green', got {result!r}"
    score += 1; print("\u2705 correctly identifies 'green' from mock response")

    # Labels are passed to the mock's prompt
    captured_prompt = {}
    def _mock_capture(b, p):
        captured_prompt['p'] = p
        return 'red'
    classify_image(_test_b64, labels, describe_fn=_mock_capture)
    assert all(l in captured_prompt['p'] for l in labels), (
        f"All labels should appear in the prompt: {captured_prompt['p']!r}")
    score += 1; print("\u2705 all labels appear in the classification prompt")

    # Fallback to labels[0] when no label found in response
    fallback = classify_image(_test_b64, labels,
                               describe_fn=lambda b, p: 'I cannot determine')
    assert fallback == labels[0], f"Expected fallback {labels[0]!r}, got {fallback!r}"
    score += 1; print("\u2705 falls back to labels[0] when no match found")

    # Works with 2-label binary classification
    two = classify_image(_test_b64, ['indoor', 'outdoor'],
                          describe_fn=lambda b, p: 'outdoor')
    assert two == 'outdoor'
    score += 1; print("\u2705 2-label classification works")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def classify_image(img_b64: str, labels: list,
                   describe_fn=None) -> str:
    label_list = ', '.join(f'"{l}"' for l in labels)
    prompt = (
        f'Classify this image into exactly one of these categories: '
        f'{label_list}. Reply with only the category name, nothing else.'
    )
    response = describe_image(img_b64, prompt, describe_fn=describe_fn)
    resp_lower = response.lower()
    for label in labels:
        if label.lower() in resp_lower:
            return label
    return labels[0]
```

**Why case-insensitive matching and a fallback?** Vision LLMs may return "Green" or "GREEN" rather than the exact string you provided. The `.lower()` comparison handles this. The `labels[0]` fallback ensures the function always returns a valid label — the caller can then check if the confidence is meaningful (e.g., flag items where the fallback was used).

</details>